In [2]:
# Ensure compatible numpy for loading pickled bin_edges
%pip install -q --upgrade numpy


[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Clear any existing PyTorch cache
# CPU inference for `NisithDissanayake/genknob-tuner`
import os, json, ast, re, pickle
import torch
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
import gc
torch.cuda.empty_cache() if torch.cuda.is_available() else None
gc.collect()

/home/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0

In [2]:
# Settings
MODEL_REPO = "NisithDissanayake/genknob-tuner"
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
N_BINS = 20  # matches training

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load PEFT-wrapped model to CPU with low memory usage
try:
    model = AutoPeftModelForCausalLM.from_pretrained(
        MODEL_REPO,
        device_map="cpu",
        low_cpu_mem_usage=True,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    model.to("cpu")
    model.eval()
    print("Model loaded on CPU.")
except Exception as e:
    print("Failed to load model on CPU:", repr(e))

Loading weights: 100%|██████████| 339/339 [00:26<00:00, 12.99it/s, Materializing param=model.norm.weight]                              


Model loaded on CPU.


In [3]:
# Parse one test row and build prompt (no pandas)
import json, ast, re, csv
from pathlib import Path

csv_path = "/home/E2ETune-AI4DB/inference_test/test_data.csv"
assert Path(csv_path).exists(), "test_data.csv not found"

with open(csv_path, newline="") as f:
    reader = csv.DictReader(f)
    first = next(reader)

# Columns that contain dict-like strings
dict_cols = [
    "best_config","internal_metrics","workload_features",
    "query_plan_features","binned_config"
 ]
for c in dict_cols:
    if first.get(c) is not None:
        first[c] = ast.literal_eval(first[c])

row = first
print("Selected benchmark:", row.get("benchmark"), "workload_idx:", row.get("workload_idx"))

N_BINS = 20

def format_instruction(r):
    metrics = r.get('internal_metrics', {})
    workload = r.get('workload_features', {})
    plan = r.get('query_plan_features', {})
    metrics_str = "\n".join([f"- {k}: {v}" for k, v in metrics.items() if float(v) > 0])
    workload_str = "\n".join([f"- {k}: {v}" for k, v in workload.items()])
    plan_str = "\n".join([f"- {k}: {v}" for k, v in plan.items()])
    user_prompt = f"""You are an expert PostgreSQL DBA. Analyze the workload state, features, and query plans below.
Determine the optimal database configuration knobs to maximize performance.
Output the configuration as a JSON object where values are Bin Indices (0-{N_BINS-1}).
\n### Workload Context ({r.get('source_benchmark', 'Unknown')}):
**Internal Metrics (State):**
{metrics_str}
\n**Workload Features (Stats):**
{workload_str}
\n**Query Plan Signals (Structure):**
{plan_str}
\n### Task:
Predict the optimal configuration bucket for each knob."""
    return f"<|im_start|>system\nYou are a database tuning assistant.<|im_end|>\n<|im_start|>user\n{user_prompt}<|im_end|>\n<|im_start|>assistant\n"

prompt = format_instruction(row)
print("Prompt prepared (truncated):\n", prompt[:400], "...\n")

Selected benchmark: job workload_idx: 107
Prompt prepared (truncated):
 <|im_start|>system
You are a database tuning assistant.<|im_end|>
<|im_start|>user
You are an expert PostgreSQL DBA. Analyze the workload state, features, and query plans below.
Determine the optimal database configuration knobs to maximize performance.
Output the configuration as a JSON object where values are Bin Indices (0-19).

### Workload Context (job):
**Internal Metrics (State):**
- xact_c ...



In [4]:
# Generate on CPU and decode prediction
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.0, do_sample=False)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Raw generation (truncated):\n", text[:800], "...\n")

# Try to extract the first JSON object from the assistant output
def extract_json(s: str):
    import json, ast, re
    m = re.search(r"\{.*?\}", s, flags=re.DOTALL)
    if not m:
        return None
    block = m.group(0)
    try:
        return json.loads(block)
    except Exception:
        try:
            return ast.literal_eval(block)
        except Exception:
            return None

pred_bins = extract_json(text)
print("Predicted bins:", pred_bins)

# Optional: decode to real values only if bin_edges is available
def decode_bins_to_real(pred_bins, bin_edges):
    if not isinstance(pred_bins, dict) or not isinstance(bin_edges, dict):
        return None
    real = {}
    for knob, idx in pred_bins.items():
        edges = bin_edges.get(knob)
        if edges is None:
            continue
        try:
            i = int(idx)
        except Exception:
            try:
                i = int(float(idx))
            except Exception:
                i = 0
        i = max(0, min(i, len(edges)-2))
        real[knob] = (edges[i] + edges[i+1]) / 2
    return real

decoded = None
try:
    if 'bin_edges' in globals() and isinstance(bin_edges, dict):
        decoded = decode_bins_to_real(pred_bins, bin_edges)
        print("Decoded real-valued config:")
        print(decoded)
    else:
        print("bin_edges not available; skipping real-value decode.")
except Exception as e:
    print("Decode skipped due to error:", repr(e))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Raw generation (truncated):
 system
You are a database tuning assistant.
user
You are an expert PostgreSQL DBA. Analyze the workload state, features, and query plans below.
Determine the optimal database configuration knobs to maximize performance.
Output the configuration as a JSON object where values are Bin Indices (0-19).

### Workload Context (job):
**Internal Metrics (State):**
- xact_commit: 7.0
- xact_rollback: 1.0
- blks_read: 56829.0
- blks_hit: 47909.0
- tup_returned: 1402794.0
- tup_fetched: 1267391.0
- disk_read_count: 292092.0
- disk_read_bytes: 2392817664.0

**Workload Features (Stats):**
- size: 2.0
- read_ratio: 1.0
- group_by_ratio: 0.0
- order_by_ratio: 0.0
- avg_query_length: 153.0
- avg_joins: 0.0
- filter_ratio: 1.0

**Query Plan Signals (Structure):**
- plan__count__aggregate: 4
- plan__count__s ...

Predicted bins: {'shared_buffers': 1, 'work_mem': 19, 'maintenance_work_mem': 19, 'effective_cache_size': 19, 'max_connections': 12, 'wal_buffers': 17, 'checkpoint_c